In [1]:
import warnings
warnings.filterwarnings('ignore')

import torch
import pandas as pd

from flonacomldft.utils.io_utils import load_pickle_file
from flonacomldft.internal_coordinates import Coordinates_mapping
from flonacomldft.utils.io_utils import save_ase_molecules_as_traj, save_pickle_file

In [2]:
path = '/mnt/home/amolina/ceph/adaptive-350/adaptive-dft'

mcmc_paths = '5-multimodal/results_multimodal_sampling_mixture_29096190/mixture_mcmc_dic_29096190.pkl'#,
             #'6-multimodal-mlp/results_multimodal_sampling_mixture_29134160/mixture_mcmc_dic_29134160.pkl',
             #'6-multimodal-mlp/results_multimodal_sampling_mixture_29134161/mixture_mcmc_dic_29134161.pkl']

mcmc = load_pickle_file(mcmc_paths, path)

In [3]:
coord_mapping = Coordinates_mapping(etype='dft')

In [4]:
xs = mcmc['xs']
us = mcmc['us']
isomers = mcmc['isomers']

In [5]:
xs[0, 1].reshape(1, -1)

tensor([[-0.1864, -0.1020,  0.0330, -0.0757, -0.0531,  0.1602,  0.2084, -0.1241,
          0.0693,  0.0309,  0.0300, -0.0596]], grad_fn=<ReshapeAliasBackward0>)

In [6]:
ij_isomers_0 = torch.stack(torch.where(isomers==0))

In [7]:
ij_isomers_1 = torch.stack(torch.where(isomers==1))

In [8]:
n_start = [49737, 63]

In [9]:
steps, chains = xs.shape[:-1]

isomers_0 = []
us_0 = []

for ij in ij_isomers_0.T[n_start[0]:]:
    print(ij)
    if isomers[ij[0], ij[1]].item() == 0:
        molecule, logdetjac = coord_mapping.build_molecule_from_real_centered(xs[ij[0], ij[1]].reshape(1, -1), isomer=isomers[ij[0], ij[1]].item())
        u = coord_mapping.compute_energy_in_new_frame(us[ij[0], ij[1]], logdetjac=logdetjac)
        isomers_0.append(molecule)
        us_0.append(u)

tensor([998,   0])
tensor([998,   1])
tensor([998,   2])
tensor([998,   3])
tensor([998,   4])
tensor([998,   5])
tensor([998,   6])
tensor([998,   7])
tensor([998,   8])
tensor([998,   9])
tensor([998,  10])
tensor([998,  11])
tensor([998,  12])
tensor([998,  13])
tensor([998,  14])
tensor([998,  15])
tensor([998,  16])
tensor([998,  17])
tensor([998,  18])
tensor([998,  19])
tensor([998,  20])
tensor([998,  21])
tensor([998,  22])
tensor([998,  23])
tensor([998,  24])
tensor([998,  25])
tensor([998,  26])
tensor([998,  27])
tensor([998,  28])
tensor([998,  29])
tensor([998,  30])
tensor([998,  31])
tensor([998,  32])
tensor([998,  33])
tensor([998,  34])
tensor([998,  35])
tensor([998,  36])
tensor([998,  37])
tensor([998,  38])
tensor([998,  39])
tensor([998,  40])
tensor([998,  41])
tensor([998,  42])
tensor([998,  43])
tensor([998,  44])
tensor([998,  45])
tensor([998,  46])
tensor([998,  47])
tensor([998,  48])
tensor([998,  49])
tensor([999,   0])
tensor([999,   1])
tensor([999,

In [10]:
isomers_1 = []
us_1 = []

for ij in ij_isomers_1.T[n_start[1]:]:
    if isomers[ij[0], ij[1]].item() == 1:
        molecule, logdetjac = coord_mapping.build_molecule_from_real_centered(xs[ij[0], ij[1]].reshape(1, -1), isomer=isomers[ij[0], ij[1]].item())
        u = coord_mapping.compute_energy_in_new_frame(us[ij[0], ij[1]], logdetjac=logdetjac)
        isomers_1.append(molecule)
        us_1.append(u)

In [20]:
ij_isomers_0[1, n_start[0]:].unique(), ij_isomers_1[1, n_start[1]:].T.unique()

(tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
         36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]),
 tensor([ 0,  1,  2,  5,  8,  9, 10, 11, 12, 17, 18, 19, 21, 22, 23, 24, 25, 26,
         27, 29, 30, 31, 33, 34, 35, 37, 38, 40, 41, 43, 44, 45, 46, 47, 48]))

In [11]:
us = {'Pot Energy' + str(i) : torch.cat(us).detach().numpy() for i, us in enumerate([us_0, us_1])}
us_df = pd.DataFrame(us)
us_df


,Pot Energy0,Pot Energy1
0,-6.290011,-6.110669
1,-6.242333,-6.398486
2,-6.319275,-6.398486
3,-6.546427,-6.394553
4,-6.423967,-6.370979
...,...,...
95,-6.428780,-6.294280
96,-6.290222,-6.293079
97,-6.278857,-6.325919
98,-6.298487,-6.428191


In [12]:
save_ase_molecules_as_traj(isomers_0, 'MCMC_isomer_0.traj')
save_ase_molecules_as_traj(isomers_1, 'MCMC_isomer_1.traj')

us_df.to_csv('MCMC_pot_energy.csv', index=False)

In [13]:
dic_mcmc_configs = {'molecules isomer 0': isomers_0,
                    'molecules isomer 1': isomers_1,
                    'potential energy': us_df}

save_pickle_file(dic_mcmc_configs, 'mcmc_configs.pkl')